<a href="https://colab.research.google.com/github/tinkerer9/imdb-collaborative-filter/blob/main/IMDB_Collaborative_Filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMDB Collaborative Filter

> <small>
<b>Author:</b> Max Parisi<br>
<b>Year:</b> 2025<br>
<b>Info:</b> Made during the <a href="https://www.bransonsummer.org/applied-ai-in-python-2">Applied AI in Python</a> Branson Summer Session with instructor Tony Pound<br>
<b>Code:</b> Python 3<br>
<b>Link:</b> <a href="https://colab.research.google.com/drive/1H-imyPHMtxoq_W3JoGZcy9bvqyZn_9GD?usp=sharing">bit.ly/imdbfilter</a>
</small>

This program uses cosine similarity to predict which movies you will like based on what movies other users on IMDB like.

### Intended use

The user may like some movies but not others and will rate them on a scale from 1 to 5 (fractions allowed). If they haven't seen the movie, they will leave the field blank.
The program will:

1.   Find similar users based off of (seen) movies
2.   Use those similar users to predict how the user would rate the unseen movies

This is known as **collaborative filtering**.

### Collaborative filtering

[Collaborative filtering](https://en.wikipedia.org/wiki/Collaborative_filtering) is a method of making predictions (filtering) about the interests of a user by collecting information from many other users (collaborative).

### Data source

The data in this project come from two CSV files from [IMDB](https://imdb.com): [`movies.csv`](https://github.com/tinkerer9/imdb-collaborative-filter/blob/main/data/movies.csv) and [`ratings.csv`](https://github.com/tinkerer9/imdb-collaborative-filter/blob/main/data/ratings.csv). If the files are not found, the program automatically downloads them. See more at [`getIMDBRatings()`](https://colab.research.google.com/drive/1H-imyPHMtxoq_W3JoGZcy9bvqyZn_9GD#scrollTo=Tasu3gTy8Wqe).

## Libraries

This script uses 7 libraries: [`math`](https://docs.python.org/3/library/math.html), [`numpy`](https://numpy.org/), [`pandas`](https://pandas.pydata.org/), [`sys`](https://docs.python.org/3/library/sys.html), [`os`](https://docs.python.org/3/library/os.html), [`requests`](https://pypi.org/project/requests/), and [`datetime`](https://docs.python.org/3/library/datetime.html).



### Uses

| Library | Use |
| :-: | :-: |
| `math` | More advanced math (e.g. square root) |
| `numpy` | Advanced lists and optimized math algorithms |
| `pandas` | Interfacing with CSV files and creating datasets |
| `sys` | `sys.stdout.write()` for advanced output |
| `os` | `os.path.exists()` to see if the CSVs exist |
| `requests` | Downloading CSVs from GitHub |
| `datetime` | [Unix time](https://en.wikipedia.org/wiki/Unix_time) conversion |


In [ ]:
import math
import numpy as np
import pandas as pd
import sys
import os
import requests
from datetime import datetime

## `getIMDBRatings()`

**Arguments:** none

**Returns:** nothing

This function parses the data from [two CSV files](https://github.com/tinkerer9/imdb-collaborative-filter/tree/main/data), `movies.csv` and `ratings.csv`.

These datasets contain <u>10329</u> movies, <u>668</u> users, and <u>105339</u> ratings from [IMDB](https://imdb.com)!

### CSV previews

#### `movies.csv`

<small>
contains 10329 movies
</small>

| movieId | title | genres |
| :-: | :-: | :-: |
| 1 | Toy Story (1995) | Adventure\|Animation\|Children\|Comedy\|Fantasy |
| 2 | Jumanji (1995) | Adventure\|Children\|Fantasy |
| 3 | Grumpier Old Men (1995) | Comedy\|Romance |
| 4 | Waiting to Exhale (1995) | Comedy\|Drama\|Romance |
| 5 | Father of the Bride Part II (1995) | Comedy |
| ... | ... | ... |
| 148238 | A Very Murray Christmas (2015) | Comedy |
| 148226 | The Big Short (2015) | Drama |
| 149532 | Marco Polo: One Hundred Eyes (2015) | (no genres listed) |

#### `ratings.csv`

<small>
contains 668 users and 105339 ratings
</small>

| userId | movieId | rating | timestamp |
| :-: | :-: | :-: | :-: |
| 1 | 16 | 4 | 1217897793 |
| 1 | 24 | 1.5 | 1217895807 |
| 1 | 32 | 4 | 1217896246 |
| 1 | 47 | 4 | 1217896556 |
| 1 | 50 | 4 | 1217896523 |
| ... | ... | ... | ... |
| 668 | 143385 | 4 | 1446388585 |
| 668 | 144976 | 2.5 | 1448656898 |
| 668 | 148626 | 4.5 | 1451148148 |

### Outdated ratings

By looking at the greatest timestamp from the CSV (`1452404919`), which is in [Unix time](https://en.wikipedia.org/wiki/Unix_time), we can calculate when the last rating was:

*Sunday, January 10, 2016 5:48:39 AM GMT*

At the time of writing (September 2025), we can see that this rating for *The Grand Budapest Hotel* was made just under **ten years ago**! The data is very outdated—see [Possible features](https://colab.research.google.com/drive/1H-imyPHMtxoq_W3JoGZcy9bvqyZn_9GD#scrollTo=BQ7N0RHJra93&line=23&uniqifier=1) below.

### Global Variables

Almost all functions in this program use **global variables**. A variable can be set as "global" by using the `global` keyword:

```
def myFunc():
  global myVariable
```
In this example, the `myVariable` variable is now in the global scope. This allows it to be set in `myFunc()` to be used elsewhere.

Read more at [w3schools.com/python/python_variables_global.asp](https://www.w3schools.com/python/python_variables_global.asp).

### Automatic download

If the files aren't detected in the Colab's storage, it automatically [downloads it from GitHub](https://github.com/tinkerer9/imdb-collaborative-filter/tree/main/data) and uploads it to the storage. See more at the [`downloadIfMissing()`](https://colab.research.google.com/drive/1H-imyPHMtxoq_W3JoGZcy9bvqyZn_9GD#scrollTo=X6-aTEinSsFr) function below.

### Possible improvement

In order to convert `ratings.csv` into a [two-dimensional](https://www.geeksforgeeks.org/python/python-using-2d-arrays-lists-the-right-way/) list, this function first converts the CSV into a [pivot table](https://en.wikipedia.org/wiki/Pivot_table). This is very ineffective, since the pivot table and list has over **6 million** elements, **98%** of which are just zeros! By switching to a different system, we can greatly improve system memory and processing speed.

In [ ]:
def getIMDBRatings():
  global userIds, userIdsUnique, movieIds, numUsers, numRatings, numMovies, movieIdsRaw, ratingsList, movieIdsByRatings, movieIdToTitle

  print("Loading users...")

  # Path of ratings and movies CSV in Colab storage
  ratingsPath = "/content/ratings.csv"
  moviesPath = "/content/movies.csv"

  # If not found, download from here:
  ratingsUrl = "https://raw.githubusercontent.com/tinkerer9/imdb-collaborative-filter/refs/heads/main/data/ratings.csv"
  moviesUrl = "https://raw.githubusercontent.com/tinkerer9/imdb-collaborative-filter/refs/heads/main/data/movies.csv"

  # read users
  downloadIfMissing(ratingsPath, ratingsUrl)
  ratingsCSV = pd.read_csv(ratingsPath)

  # read movies
  downloadIfMissing(moviesPath, moviesUrl)
  moviesCSV = pd.read_csv(moviesPath)

  userIds = ratingsCSV["userId"]
  userIdsUnique = userIds.unique().tolist() # remove duplicates
  userIds = userIds.tolist() # make userIds a list
  movieIdsUnsorted = ratingsCSV["movieId"] # rated movies only
  movieIds = sorted(movieIdsUnsorted.unique())
  movieIdsByRatings = movieIdsUnsorted.value_counts().index.tolist()
  numUsers = len(userIdsUnique)
  numRatings = len(userIds)
  numMovies = len(movieIds)

  movieIdsRaw = moviesCSV["movieId"].tolist() # corresponds to below
  movieTitles = moviesCSV["title"].tolist()
  movieIdToTitle = dict(zip(movieIdsRaw, movieTitles))

  # create a pivot table
  # (row = userId, column = movieId, cell = rating)
  ratings = ratingsCSV.pivot(
    index="userId",
    columns="movieId",
    values="rating"
  ).fillna(0) # replace n/a values with a 0 (not seen)

  ratingsList = ratings.values.tolist()

  # print time of most recent rating
  maxTimestamp = max(ratingsCSV["timestamp"])
  dt_object = datetime.fromtimestamp(maxTimestamp)
  print(f"Most recent rating: {dt_object.strftime('%B %Y')}")

  print(f"{numMovies} movies, {numUsers} users, and {numRatings} ratings loaded.\n")

## `downloadIfMissing()`

**Arguments:** `path`, `url`

**Returns:** nothing

This function first finds if a file exists using `os.path.exists()` and decides what to accordingly:

- **File exists:** the function ends and the program continues.
- **File does not exist:** the function downloads the file from the `url` argument and uploads it to the Colab's storage (using the `path` argument).

### Use

This function is used in `getIMDBRatings()`, since sharing this Colab or even restarting the session [deletes the files](https://stackoverflow.com/questions/65272465/google-colab-files-are-gone). `getIMDBRatings()` uses this function to download the [CSVs from GitHub](https://github.com/tinkerer9/imdb-collaborative-filter/tree/main/data).

In [ ]:
def downloadIfMissing(path, url):
  if not os.path.exists(path): # only run if the file doesn't exist
    os.makedirs(os.path.dirname(path), exist_ok=True) # make sure folder exists
    response = requests.get(url)
    response.raise_for_status() # throw error if download fails
    with open(path, "wb") as f:
      f.write(response.content)

## `getTitleFromId()`

**Arguments:** `movieId`

**Returns:** the title of a movie based on the movie's ID

This function returns the title of a movie with the movie's ID as an input. By using the [`.get()`](https://www.w3schools.com/python/ref_dictionary_get.asp) method, this function returns the value of the purpose-built `movieIdToTitle` dictionary.

In [ ]:
def getTitleFromId(movieId):
  global movieIdToTitle

  return movieIdToTitle.get(movieId)

## `predictUser()`

**Arguments:** none

**Returns:** nothing

This function asks the user to rate the movies (using `getUserRatings()`) and then looks for similarities with other users. If the user hasn't seen a movie (rates the movie as a 0) then the program uses the ratings given by similar users and uses them to calculate a predicted rating.

### Progress bar

This function displays a "progress bar" in the program output that looks like

```
[ ██████████--------------- ]  267/668 (40%)
```
when calculating similarities by using `sys.stdout.write()`, `sys.stdout.flush()`, and `"\r"`. It is also used in the `printRecomendations()` function.

### Average similarity

The function also stores the (non-weighted) average of all similarities, `averageSimilarity`. It ranges from 0 to 1, with 0 meaning that the user is similar with absolutely no one and 1 meaning the user is 100% similar with everyone. The latter is impossible, since the user can't have the exact same rating as all other users at the same time.

In [ ]:
def predictUser():
  global ratingsList, similarities

  similarities = []

  # Get user ratings
  userRatings = getUserRatings()

  print("\nCalculating similarities...")

  progressBarLength = 25

  i = 0
  sys.stdout.write(f"\r[ {'-' * progressBarLength} ]  0/{numUsers} (0%)")
  sys.stdout.flush()

  # check to see if a similar user exists and append to a similarities list
  for otherUser in ratingsList:
    progress = i / numUsers
    progressBar = math.ceil(progress * progressBarLength)
    similarities.append(similarity(userRatings, otherUser))

    i += 1
    sys.stdout.write(f"\r[ {'█' * progressBar}{'-' * (progressBarLength-progressBar)} ]  {i}/{numUsers} ({round(progress * 100, 1)}%)")
    sys.stdout.flush()

  print(f"\r[ {'█' * progressBarLength} ]  {numUsers}/{numUsers} (100%)")
  print("Similarities calculated.")

  if sum(similarities) == 0:
    print("\nNo similar users found; unable to generate recommedations")
  else:
    printRecommendations()

## `getUserRatings()`

**Arguments:** none

**Returns:** `userRatings` (an array of one rating per movie)

This function gets ratings from a user ranging from 1 to 5. Fractions are accepted as well.

If the field was left blank, the function will take that as a `0`, meaning that the movie hasn't been seen.

### Rating meanings:

| Rating | Meaning |
| :-: | :-: |
| *blank* | Not seen |
| 1 | Hate it |
| 2 | Dislike it |
| 3 | Neutral |
| 4 | Like it |
| 5 | Love it |

### Error handling

If the user enters an invalid input, then the program could crash. This function implements a few simple checks to make sure the program can continue successfully.

#### Invalid rating

If an invalid input (e.g. `6` or `e`) is given, the function will let the user know their input was invalid and will make them try again. This function uses a [try-except](https://www.w3schools.com/python/python_try_except.asp) block when converting the user input into a float. If the try-except fails, then the user gets notified and the program can continue.

#### Early stop

While the function recomends that the user rates **50 movies**, the user can decide to stop early by typing `stop`. However, if the user stops without any movies rated, the program will have no data to make recomendations from and will return random recomendations. This function makes sure `ratedMovieIds` contains data before stopping.

#### Early edit

Users often press enter out of habit, so an editing feature was added. If they want to go back and edit the previous movie(s), they can type `back`. However, if they try to edit the movie on the very first entry, the program will overflow. To avoid this, the function checks that `i` is greater than 0.

#### Missing movie

If the movie is in `ratings.csv` but not in `movies.csv`, then the function will ask for a recomendation but won't know the movie name. If the function encounters this error, then it will skip that movie and will continue with the rest.

In [ ]:
def getUserRatings():
  global numMovies, movieIds, movieIdsByRatings, ratedMovieIds, movieIdToTitle

  ratedMovieIds = []
  userRatings = [0] * numMovies

  numMovieIdsToRate = min(50, numMovies)  # cap at total number of movies

  print(f"Rate {numMovieIdsToRate} movies 1-5, with 1 meaning you hate it, 5 meaning you love it, or leave blank if not seen. Fractions accepted.")
  print(f"Type \"stop\" to stop early, or rate {numMovieIdsToRate} movies (recommended).")
  print("If you want to edit a rating, type \"back\" to edit the previous rating.\n")

  i = 0
  while len(ratedMovieIds) < numMovieIdsToRate and i < len(movieIdsByRatings):
    movieId = movieIdsByRatings[i]

    movieTitle = movieIdToTitle.get(movieId)

    if movieTitle is None: # skip movies not in CSV
      i += 1
      continue

    rating = input(f"{len(ratedMovieIds)+1}) \t{movieTitle}: ")

    if rating.strip() == "":
      i += 1
      continue

    if rating.lower() == "stop":
      if ratedMovieIds:
        print("Stopping with ratings...")
        return userRatings
      else:
        print("You must rate at least one movie before stopping.\n")
        continue

    if rating.lower() == "back":
      if i > 0: # not if on first rating
        i -= 1 # go back to previous movie
      else:
        print("Unable to edit the previous rating.\n")
      continue
    else:
      edit = False

    try:
      rating_val = float(rating)
    except ValueError: # not a (float) number
      print("Invalid input. Please rate 1-5 or leave blank.\n")
    else:
      if 1 <= rating_val <= 5:
        idx = movieIds.index(movieId)
        userRatings[idx] = rating_val
        ratedMovieIds.append(movieId)
        i += 1
      else: # not between 1 and 5 (inclusive)
        print("Invalid input. Please rate 1-5 or leave blank.\n")

  return userRatings

## `printRecomendations()`

**Arguments:** none

**Returns:** nothing

This function finds what the user would rate their unseen movies based off of other users' ratings and prints it out.

### Improvements

In a previous version, this function would find just **one** most similar user and use their ratings to predict the current user's ratings. It didn't take other users' ratings into account.

Now, this function finds a [weighted average](https://en.wikipedia.org/wiki/Weighted_arithmetic_mean) on each movie weighted on a user's similarity. The benefits of this are more accurate ratings with the tradeoff of computing speed.

This function then lists 10 recomended (unrated) movies in descending order of `averageMovieRatings`. The user can request more recomendations.

<small>
It's important to note that <code>averageMovieRatings</code> is a weighted average.
</small>

In [ ]:
def printRecommendations():
    global similarities, numUsers, userIds, numMovies, ratingsList, manualRatings, ratedMovieIds

    print("\nCalculating recommendations...")

    averageMovieRatings = np.zeros(numMovies, dtype=float)

    for i in range(numUsers):
      progress = i / numUsers
      progressBar = math.ceil(progress * 25)

      sys.stdout.write(f"\r[ {'█' * progressBar}{'-' * (25-progressBar)} ]  {i}/{numUsers} ({round(progress * 100, 1)}%)")
      sys.stdout.flush()

      userSimilarity = similarities[i]

      if userSimilarity != 0:
        movieRatings = np.array(ratingsList[i])  # ratings for all movies by user i
        averageMovieRatings += movieRatings * userSimilarity

    sortedMovieIds = sorted(range(numMovies), key=lambda i: averageMovieRatings[i], reverse=True)
    sortedAverageMovieRatings = [averageMovieRatings[i] for i in sortedMovieIds]

    print(f"\r[ {'█' * 25} ]  {numUsers}/{numUsers} (100%)")
    print("Recommendations calculated.")

    print("\nBased off of people like you, I think you would like:\n")

    i = 0
    moviesTorecomend = 10
    recomendedMovies = 0

    while True:
      movieId = sortedMovieIds[i]

      if recomendedMovies >= moviesTorecomend:
        while True:
          userInput = input("\nShow more recommendations? (y/n): ")
          if userInput in ["y", "n"]:
            break
          else:
            print("Invalid input.")

        if userInput == "y":
          print("Showing 10 more recommendations...\n")
          moviesTorecomend += 10
        elif userInput == "n":
          print("Stopping program.")
          break

      if movieId in ratedMovieIds:
        i += 1
        continue # skip this movie

      movieTitle = getTitleFromId(movieId)

      if movieTitle != None: # movieId is in movies CSV
        print(f"{recomendedMovies + 1}) \t{getTitleFromId(movieId)}")
        recomendedMovies += 1

      i += 1

## `similarity()`

**Arguments:** 2 vectors as lists, `a` and `b`

**Returns:** the cosine similarity (`cs`)

This function uses [dot product math](https://en.wikipedia.org/wiki/Dot_product) to return a number between 0 and 1. The higher the number, the more similar another user's ratings are to the user's. In mathematical terms, this function is predicting how similar two vectors or row matrices are to each other based on the cosine of the angle between the vectors (see diagram below).

<img src="https://builtin.com/sites/www.builtin.com/files/styles/ckeditor_optimize/public/inline-images/2_cosine-similarity.png" width=25%>

To calculate the cosine similarity from two matrices, we use **dot product math** and **vector magnitude**.

For the following examples, we will be using Sophie's data (from the previous version) and fake test user data:

| Sophie | Test User |
|:-:|:-:|
| 0 | 1 |
| 3 | 4 |
| 5 | 5 |
| 1 | 0 |
| 1 | 2 |

### Dot product math

The dot product is calculated by multiplying each corresponding element in the lists and adding them together:

$$
ab
=
\begin{bmatrix} a_1 \\ a_2 \\ a_3 \\ a_4 \\ a_5 \end{bmatrix}
\cdot
\begin{bmatrix} b_1 \\ b_2 \\ b_3 \\ b_4 \\ b_5 \end{bmatrix}
=
a_1 b_1 + a_2 b_2 + a_3 b_3 + a_4 b_4 + a_5 b_5
$$

Substituting in Sophie's ratings on the left side and test user ratings on the right side, we get

$$
ab
=
\begin{bmatrix} 4 \\ 5 \\ 2 \end{bmatrix}
\cdot
\begin{bmatrix} 3 \\ 5 \\ 1 \end{bmatrix}
=
(4\cdot3) + (5\cdot5) + (2\cdot1)
$$

which is equal to $12+25+2=39$. Now we know that $ab=39$.

<small>
You'll notice that our matrices changed. If there is a zero in one list, it's corresponding element (other list) must be made zero as well. This is important in the next step, since e.g. 0<sup>2</sup> ≠ 1<sup>2</sup>. All corresponding elements must be removed.
</small>

### Vector magnitude
Vector magnitude is the "length" of the vector and is calculated by taking the square root of the sum of the squared elements:

$$
aa
=
\sqrt{\left(a_1^2 + a_2^2 + a_3^2 + a_4^2 + a_5^2 \right)}
$$

Substituting in the test user ratings and Sophie's ratings in separate equations, we get both magnitudes:

$$
aa=
\sqrt{\left(4^2 + 5^2 + 2^2 \right)}
$$

$$
bb=
\sqrt{\left(3^2 + 5^2 + 1^2 \right)}
$$

Calculating $aa$ and $bb$, we know $aa=\sqrt{45}\approx6.71$ and $bb=\sqrt{35}\approx5.92$.

<small>
Remember when I told you about the zeros? In vector magnitude, this matters a lot and will yield different results.
</small>

### Cosine similarity

We can calculate the cosine similarity ($cs$) by dividing the dot product of the vectors by the product of their magnitudes:

$$
cs
=
\frac{ab}{aa\cdot bb}
$$

Plugging in our numbers, we get

$$
cs=
\frac{39}{\sqrt{45}\cdot\sqrt{35}}
\approx 0.9827
$$

This checks out, because our test data was the same.

$cs\approx0.9827\:/\:98.27\%$

This is exactly what (the last version of) the program gives us!

**In this version, instead of 5 movies, this program uses 10325!**

### Improvements

This version has several improvements:

- Raises an error if the arrays have different shapes
- Uses [`numpy`](https://numpy.org/) for:
  - Arrays (`np.array(a)`)
  - Dot product calculation (`np.dot(a, b)`)
  - Vector magnitude (`np.linalg.norm(a)`)
- Uses a mask to filter `a` and `b`
- One-line return filter
  - Filters if `aa` and `bb` are both zero (would otherwise throw a `ZeroDivisionError` error)

These improvements greatly increase speed, but more could be made (see [Possible features](https://colab.research.google.com/drive/1H-imyPHMtxoq_W3JoGZcy9bvqyZn_9GD#scrollTo=BQ7N0RHJra93&line=23&uniqifier=1))

In [ ]:
def similarity(a, b):
  a = np.array(a)
  b = np.array(b)

  if a.shape != b.shape:
    raise ValueError(f"Arrays have different shapes: {a.shape} vs {b.shape}")

  # Create mask: True where either (or both) vectors are zero
  mask = ((a != 0) | (b != 0))

  filteredA = a[mask]
  filteredB = b[mask]

  # If all filtered out, similarity is 0
  if filteredA.size == 0:
    return 0.0

  ab = np.dot(filteredA, filteredB)
  aa = np.sqrt(np.sum(filteredA ** 2))
  bb = np.sqrt(np.sum(filteredB ** 2))

  return float(ab / (aa * bb)) if (aa != 0 and bb != 0) else 0.0

## `main()`

**Arguments:** none

**Returns:** nothing

This short function calls just two other functions, but is the backbone of the whole program.

### Anti-import

[Right after this function](https://colab.research.google.com/drive/1H-imyPHMtxoq_W3JoGZcy9bvqyZn_9GD#scrollTo=rGV59qlkpFDT&line=1&uniqifier=1), we have
```
if __name__ == "__main__":
  main()
```
which calls the `main()` function if the program itself has been run, but not if the program has been imported by checking `__name__`.

Learn more at [geeksforgeeks.org/python/what-does-the-if-\_\_name\_\_-\_\_main\_\_-do](https://www.geeksforgeeks.org/python/what-does-the-if-__name__-__main__-do/).

### Loading optimization

`getIMDBRatings()` only needs to be called once. If the function wasn't yet called, then all variables will not be defined. To optimize data loading speed, the `main()` function uses a try-except block to: check if `userIds` is defined, and if not, calls `getIMDBRatings()`. This greatly increases speed, as loading and processing all 105339 ratings takes quite a long time to do every single time the program is run!

### Cell stopping

When the code cell is stopped, the program will throw a `KeyboardInterrupt` error. To stop this error from happening, this function uses a try-except block to detect if the cell is stopped and then uses [`pass`](https://www.w3schools.com/python/ref_keyword_pass.asp) to do nothing and stop the program without throwing an error.

In [ ]:
def main():
  try:
    # get ratings from CSVs
    try: # check if userIds is defined
      userIds # could be any variable defined by getIMDBRatings()
    except NameError: # data not loaded
      getIMDBRatings() # load data

    # Find most similar user and predict ratings if applicable
    predictUser()
  except KeyboardInterrupt: # gets triggered when a cell is stopped
    pass # do nothing / end program without error

## Program output

Run this program differently depending on whether you've used it before:

- **It's my first time running this program:**
  - Click `Run all` at the top of the screen
  - Not doing this  results in this error: `NameError: name 'main' is not defined`

- **I've just run this program:**
  - Click the play icon in the cell below.

**All instructions are below:**

In [ ]:
if __name__ == "__main__":
  main()

Loading users...
Most recent rating: January 2016
10325 movies, 668 users, and 105339 ratings loaded.

Rate 50 movies 1-5, with 1 meaning you hate it, 5 meaning you love it, or leave blank if not seen. Fractions accepted.
Type "stop" to stop early, or rate 50 movies (recommended).
If you want to edit a rating, type "back" to edit the previous rating.

1) 	Pulp Fiction (1994): 
1) 	Forrest Gump (1994): 
1) 	Shawshank Redemption, The (1994): 
1) 	Jurassic Park (1993): 5
2) 	Silence of the Lambs, The (1991): 
2) 	Star Wars: Episode IV - A New Hope (1977): 
2) 	Matrix, The (1999): 
2) 	Terminator 2: Judgment Day (1991): 
2) 	Braveheart (1995): 
2) 	Schindler's List (1993): 
2) 	Fugitive, The (1993): 
2) 	Toy Story (1995): 
2) 	Usual Suspects, The (1995): 
2) 	Star Wars: Episode V - The Empire Strikes Back (1980): 
2) 	Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981): 
2) 	Star Wars: Episode VI - Return of the Jedi (1983): 
2) 	Batman (1989): 
2) 	American Beau

## Future changes

This program was a great learning experience, but it could still be made much better. Below are many possible changes, with more important ones bolded.

- **Use [NumPy arrays](https://numpy.org/doc/stable/reference/generated/numpy.array.html) *everywhere* instead of just in the `similarity()` function**
  - A lot faster, especially with more data.
  - Much more support than with lists
  - Allows for a less complex program
- **More data!**
  - More data makes better results
  - 668 users isn't *that* many for a machine learning program like this
  - Very outdated—the last rating in the CSV was from 2016!—see [`getIMDBRatings()`](https://colab.research.google.com/drive/1H-imyPHMtxoq_W3JoGZcy9bvqyZn_9GD#scrollTo=Tasu3gTy8Wqe&line=75&uniqifier=1)
  - I tried doing this with a [MovieLens dataset containing 32 million ratings](https://grouplens.org/datasets/movielens/32m/), but with my current (inefficient) code, both hosted and local runtimes didn't have enough memory.
  - Fetch data from the [IMDB API](https://developer.imdb.com/) (AWS only, expensive)
  - Incorporate data from multiple sources ([IMDB](https://imdb.com), [Rotten Tomatoes](https://www.rottentomatoes.com/), etc...)
- Make a [GUI](https://en.wikipedia.org/wiki/Graphical_user_interface)
  - [`tkinter`](https://docs.python.org/3/library/tkinter.html) (doesn't work on Google Colab)
  - Best option: website (needs to be programmed in Javascript/HTML, not Python — server?)
- Make more user friendly
- Clear rating-input line if left blank (shows exactly 50 lines even with unseen movies)
- Show images and genres in recomendations
  - Images: needs a different CSV / [API](https://en.wikipedia.org/wiki/API
  )
- 2D list alternative (see [`getIMDBRatings()`](https://colab.research.google.com/drive/1H-imyPHMtxoq_W3JoGZcy9bvqyZn_9GD#scrollTo=Tasu3gTy8Wqe&line=77&uniqifier=1))